In [ ]:
import numpy as np
import pyceres as crs
import rerun as rr
import scipy.spatial.transform as spt
import wrenfold as wf
from wrenfold.geometry import Quaternion

In [ ]:
# Generate some random rotations
num_obs = 50
max_deg = 30
Rs = spt.Rotation.from_euler(
    "xyz", np.random.rand(num_obs, 3) * max_deg, degrees=True
) * spt.Rotation.from_euler("xyz", [10, 20, 30], degrees=True)


In [ ]:
def rotation_error(q_obs_xyzw: wf.Vector4, q_est_xyzw: wf.Vector4):
    """
    Tangent-space difference between two scalar-last quaternions, scaled by
    a positive weight value.
    """
    q_obs = Quaternion.from_xyzw(q_obs_xyzw)
    q_est = Quaternion.from_xyzw(q_est_xyzw)

    error = (q_obs.conjugate() * q_est).to_rotation_vector(epsilon=1.0e-16)
    jac = error.jacobian(q_est_xyzw)

    return [
        wf.ReturnValue(error),
        wf.OutputArg(jac, name="jac", is_optional=True),
    ]


rotation_error_wf, gen_code = wf.generate_python(
    rotation_error, wf.PythonGenerator(use_output_arguments=True)
)
print(gen_code)

In [ ]:
class RotationAverageError(crs.CostFunction):
    def __init__(self, R_obs: spt.Rotation):
        super().__init__()
        self.set_num_residuals(3)
        self.set_parameter_block_sizes([4])
        self.q_obs = R_obs.as_quat()

    def Evaluate(
        self,
        parameters: list[np.ndarray],
        residuals: np.ndarray,
        jacobians: list[np.ndarray] | None,
    ) -> bool:
        jacs = [None] * self.num_parameter_blocks() if jacobians is None else jacobians
        residuals[:] = np.ravel(rotation_error_wf(self.q_obs, *parameters, *jacs))
        return True

In [ ]:
problem = crs.Problem()
loss = crs.TrivialLoss()

quat_ave = spt.Rotation.identity().as_quat()

for R_obs in Rs:
    cost = RotationAverageError(R_obs)
    _ = problem.add_residual_block(cost, loss, [quat_ave])

problem.set_manifold(quat_ave, crs.EigenQuaternionManifold())

options = crs.SolverOptions()
options.linear_solver_type = crs.LinearSolverType.DENSE_QR
options.minimizer_progress_to_stdout = True


summary = crs.SolverSummary()
crs.solve(options, problem, summary)
print(summary.BriefReport())

In [ ]:
rr.init("rotation average")

for i, R in enumerate(Rs):
    rr.log(str(i), rr.Transform3D(mat3x3=R.as_matrix()), rr.TransformAxes3D(0.5), static=True)

rr.log(
    "average",
    rr.Transform3D(mat3x3=spt.Rotation.from_quat(quat_ave).as_matrix()),
    rr.TransformAxes3D(1.0),
    static=True,
)
rr.notebook_show()